# Empirical Study #1: Qwen2.5-0.5B 의 factual QA 정확도는 temperature 전 구간에서 <2pp 편차를 보인다

**Hypothesis:** Qwen2.5-0.5B-Instruct 를 TriviaQA rc.nocontext 100 문항에 대해 temperature {0.0, 0.3, 0.7, 1.0} 로 평가했을 때, 전 구간 exact-match accuracy 의 max-min spread 가 2 퍼센트포인트 미만일 것이다. 즉, <1B 파라미터 급의 작은 OSS LLM 에서는 factual recall 이 sampling temperature 에 민감하지 않다.

**Rationale:** OpenAI / Anthropic 같은 대형 모델에서는 temperature 가 hallucination rate 를 명확히 바꾸는 것이 반복 관측돼 왔지만, 1B 미만 소형 OSS 모델에서는 weight 자체의 factual recall 한계가 상한을 덮어써 temperature 민감도가 사라질 것이라 추정한다. 가설이 지지되면 소형 모델 배포 시 temperature 튜닝에 쓰는 시간을 절약할 수 있고, 기각되면 small-LLM 튜닝 가이드에 '사실 정확도를 원하면 greedy' 규칙을 명문화해야 한다.

**Independent variable:** `temperature` — values `[0.0, 0.3, 0.7, 1.0]`

**Dependent metric:** `exact_match_accuracy`

---

*Generated by Vivory Research Empirical Pipeline.*
*Produced: 2026-05-21 06:51 UTC*


## 1. Setup


In [ ]:
# Install — CPU/GPU 공용. Kaggle 기본 환경에 거의 있음.
!pip install -q transformers==4.44.2 accelerate==0.34.2 datasets==3.0.0 torch


## 2. Configuration


In [ ]:
import json

CONFIG = {
  "study_id": 1,
  "independent_var": "temperature",
  "values": [
    0.0,
    0.3,
    0.7,
    1.0
  ],
  "dependent_metric": "exact_match_accuracy",
  "baseline_config": {
    "temperature": 0.0,
    "top_p": 1.0
  },
  "model_id": "Qwen/Qwen2.5-0.5B-Instruct",
  "dataset": "trivia_qa",
  "dataset_split": "validation[:100]",
  "num_samples": 100,
  "experiment_kind": "llm_temperature_sweep"
}

print('Study:', CONFIG['study_id'])
print('Sweep variable:', CONFIG['independent_var'], '=', CONFIG['values'])
print('Metric:', CONFIG['dependent_metric'])


## 3. Run sweep


In [ ]:
import torch, re
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset

MODEL_ID = 'Qwen/Qwen2.5-0.5B-Instruct'
DATASET = 'trivia_qa'
DATASET_SPLIT = 'validation[:100]'
NUM_SAMPLES = 100

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16 if device == 'cuda' else torch.float32,
    device_map=device,
)
model.eval()

# TriviaQA rc.nocontext subset — no-context QA
ds = load_dataset(DATASET, 'rc.nocontext', split=DATASET_SPLIT)
ds = ds.select(range(min(NUM_SAMPLES, len(ds))))
print(f'Loaded {len(ds)} questions')

def normalize(s: str) -> str:
    s = s.lower().strip()
    s = re.sub(r'[^a-z0-9 ]', '', s)
    s = re.sub(r'\s+', ' ', s)
    return s

def exact_match(pred: str, answers: list) -> int:
    p = normalize(pred)
    for a in answers:
        if normalize(a) in p or p in normalize(a):
            return 1
    return 0

def run_one_temperature(temp: float) -> dict:
    correct = 0
    for ex in ds:
        q = ex['question']
        answers = ex['answer']['aliases'] + [ex['answer']['value']]
        messages = [
            {'role': 'system', 'content': 'Answer the question concisely. No explanation.'},
            {'role': 'user', 'content': q},
        ]
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(prompt, return_tensors='pt').to(device)
        with torch.no_grad():
            if temp == 0.0:
                outputs = model.generate(**inputs, max_new_tokens=32, do_sample=False,
                                          pad_token_id=tokenizer.eos_token_id)
            else:
                outputs = model.generate(**inputs, max_new_tokens=32, do_sample=True,
                                          temperature=temp, top_p=0.95,
                                          pad_token_id=tokenizer.eos_token_id)
        pred = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
        correct += exact_match(pred, answers)
    return {'accuracy': correct / len(ds), 'n': len(ds), 'correct': correct}

# Sweep
results = {}
for temp in CONFIG['values']:
    print(f'\n▶ temperature={temp}')
    r = run_one_temperature(temp)
    results[str(temp)] = r
    print(f'  accuracy = {r["accuracy"]:.4f} ({r["correct"]}/{r["n"]})')

# Baseline = temperature=0.0
baseline_key = str(CONFIG['baseline_config'].get('temperature', 0.0))
BASELINE_METRICS = {'accuracy': results.get(baseline_key, {}).get('accuracy', 0)}
VARIANT_METRICS = [{'config': {'temperature': float(k)}, 'metrics': v} for k, v in results.items()]


## 4. Save metrics.json


In [ ]:
import json

verdict = None
verdict_confidence = None

# 간단한 verdict 자동 판정 — 독립변수 전체 구간의 accuracy range
try:
    accs = [v['metrics'].get('accuracy') for v in VARIANT_METRICS
            if v.get('metrics', {}).get('accuracy') is not None]
    if accs:
        spread = max(accs) - min(accs)
        # 스웨프 전체 스프레드 < 0.02 → 가설 지지 (temperature-robust)
        if spread < 0.02:
            verdict = 'hypothesis_supported'
            verdict_confidence = max(0.5, 1.0 - spread / 0.02)
        elif spread < 0.04:
            verdict = 'inconclusive'
            verdict_confidence = 0.5
        else:
            verdict = 'hypothesis_rejected'
            verdict_confidence = min(1.0, spread / 0.08)
except Exception as e:
    print('Verdict auto-judge failed:', e)

out = {
    'study_id': CONFIG['study_id'],
    'baseline_metrics': BASELINE_METRICS,
    'variant_metrics': VARIANT_METRICS,
    'verdict': verdict,
    'verdict_confidence': verdict_confidence,
    'independent_var': CONFIG['independent_var'],
    'dependent_metric': CONFIG['dependent_metric'],
}

with open('metrics.json', 'w') as f:
    json.dump(out, f, indent=2)

print(json.dumps(out, indent=2))
